In [ ]:
import illustris_python as il
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


basePath = "/Users/users/roana/roana/BSc_Thesis/TNG100-3/output"

gas = il.snapshot.loadSubset(basePath, 99, 'gas', fields = ['Coordinates','Masses', "Velocities"])

coords = gas['Coordinates']
masses = gas['Masses']
vels = gas["Velocities"]

In [ ]:
z_min = 30000
z_max = 45000

mask = (coords[:,2] > z_min) & (coords[:,2] < z_max)
x = coords[:, 0][mask]
y = coords[:, 1][mask]
#m = masses[mask]

plt.figure(figsize = (10, 8))

plt.hist2d(x, y, bins = 500, cmap = "viridis", norm = LogNorm())
plt.colorbar(label='Gas Mass (code units)')
plt.xlabel('X (ckpc/h)')
plt.ylabel('Y (ckpc/h)')
plt.title('TNG100-3 Cosmic Web - Gas')
#plt.xlim(10000,30000)
#plt.ylim(40000,60000)

plt.show()

In [ ]:
stars = il.snapshot.loadSubset(basePath, 99, 'stars', fields = ['Coordinates','Velocities',"Masses"])
stars_coords = stars['Coordinates']
stars_vels = stars['Velocities']
stars_masses = stars['Masses']

x = stars_coords[:, 0]
y = stars_coords[:, 1]

plt.figure(figsize = (10, 8))

plt.hist2d(x, y, bins = 1000, cmap = "twilight", norm = LogNorm())
plt.colorbar(label='Gas Mass (code units)')
plt.xlabel('X (ckpc/h)')
plt.ylabel('Y (ckpc/h)')
plt.title('TNG100-3 Cosmic Web - Stars')
#plt.xlim(10000,30000)
#plt.ylim(40000,60000)

plt.show()

In [ ]:
dm = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['Coordinates','Velocities'])
dm_coords = dm['Coordinates']
dm_vels = dm['Velocities']

x = dm_coords[:, 0]
y = dm_coords[:, 1]

plt.figure(figsize = (10, 8))

plt.hist2d(x, y, bins = 500, cmap = "inferno", norm = LogNorm())
plt.colorbar(label='Gas Mass (code units)')
plt.xlabel('X (ckpc/h)')
plt.ylabel('Y (ckpc/h)')
plt.title('TNG100-3 Cosmic Web - Dark Matter')
#plt.xlim(10000,30000)
#plt.ylim(40000,60000)

plt.show()

In [ ]:
z_min = 30000
z_max = 45000


plt.figure(figsize = (10, 8))


mask = (dm_coords[:,2] > z_min) & (dm_coords[:,2] < z_max)
x = dm_coords[:, 0][mask]
y = dm_coords[:, 1][mask]

plt.hist2d(x, y, bins = 500, cmap = "Grays", norm = LogNorm(), label = "Dark Matter")

mask = (coords[:,2] > z_min) & (coords[:,2] < z_max)

x = coords[:, 0][mask]
y = coords[:, 1][mask]

#plt.hist2d(x, y, bins = 500, cmap = "hot", norm = LogNorm(), label ="Gas", alpha=0.1)


plt.xlabel('X (ckpc/h)')
plt.ylabel('Y (ckpc/h)')
plt.title('TNG100-3 Cosmic Web')
#plt.xlim(10000,30000)
#plt.ylim(40000,60000)

plt.show()

In [ ]:
dm_vels_abs = np.sqrt(dm_vels[:,0]**2 + dm_vels[:,1]**2 + dm_vels[:,2]**2)

In [ ]:
plt.hist(dm_vels_abs, bins=100)
plt.grid()
plt.show()


In [ ]:
gas_vels_abs = np.sqrt(vels[:,0]**2 + vels[:,1]**2 + vels[:,2]**2)

In [ ]:
plt.hist(gas_vels_abs, bins=100)
plt.grid()
plt.show()

In [ ]:
stars_vels_abs = np.sqrt(stars_vels[:,0]**2 + stars_vels[:,1]**2 + stars_vels[:,2]**2)

In [ ]:
plt.hist(stars_vels_abs, bins=100)
plt.grid()
plt.show()

In [ ]:
from scipy.constants import k
k_B = k

def M_B_distrib(v,T):
# 1. Handle the constant part: sqrt(2/pi) * (m / (k_B * T))^(3/2)
    # Note: Your original formula was missing the 3/2 power for 3D MB, 
    # but I'll stick to your provided structure while keeping it safe.
    
    # Pre-calculate the exponent factor to check for overflows
    exponent_factor = (m * v**2) / (2 * k_B * T)
    
    # 2. Work in log-space to prevent v**2 or exp() from hitting inf
    term1 = 0.5 * np.log(2/np.pi)
    term2 = np.log(m / (k_B * T))
    term3 = 2 * np.log(v)
    term4 = -exponent_factor
    
    log_result = term1 + term2 + term3 + term4
    
    # 3. Use np.exp() only at the very end
    # Optional: clip the log_result to avoid exp(700+) which is the float64 limit
    return np.exp(np.clip(log_result, -700, 700))

In [ ]:
stars_masses = stars_masses * 10**10 * 2 * 10**30 #convert to kg
stars_vels_abs = stars_vels_abs * 1000 #convert to m/s

In [ ]:
m = np.exp(np.average(np.log(stars_masses)))
print (m)

In [ ]:
counts, bin_edges, patches = plt.hist(stars_vels_abs, bins=500, alpha=0.5, color='blue', density = True)
plt.grid()

bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

plt.scatter(bin_centers, counts, color='red', marker='o')

In [ ]:
from scipy.optimize import curve_fit

In [ ]:
popt, pcov = curve_fit(M_B_distrib, bin_centers, counts, p0=(1500))
print (popt)

In [ ]:
xx = bin_centers
yy = M_B_distrib(xx, 150)

plt.plot(xx,yy)
plt.show()